In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyodbc as odbc

In [2]:
conexion = odbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=.\\SQLEXPRESS;"
    "DATABASE=DBVentas;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

In [3]:
query = "SELECT * FROM RegistroVentas"

df = pd.read_sql(query, conexion)
df.head()

C:\Users\mateo\AppData\Local\Temp\ipykernel_13716\72049570.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexion)


,ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Product_ID,Sales,Quantity,Profit,Returns,Payment_Mode
0,4918,CA-2019-160304,2019-01-01,2019-01-07,Standard Class,BM-11575,FUR-BO-10004709,7394.0,1,2826.68,#N/A,Online
1,4919,CA-2019-160304,2019-01-02,2019-01-07,Standard Class,BM-11575,FUR-BO-10004709,17394.0,3,3826.68,#N/A,Online
2,4920,CA-2019-160304,2019-01-02,2019-01-07,Standard Class,BM-11575,TEC-PH-10000455,23198.0,2,6727.42,#N/A,Cards
3,3074,CA-2019-125206,2019-01-03,2019-01-05,First Class,LR-16915,OFF-ST-10003692,11446.0,2,286.15,#N/A,Online
4,8604,US-2019-116365,2019-01-03,2019-01-08,Standard Class,CA-12310,TEC-AC-10002217,3008.0,2,-52.64,#N/A,Online


In [5]:
query2= """
    SELECT * FROM Customers
"""

df=pd.read_sql(query2, conexion)
df.head()

C:\Users\mateo\AppData\Local\Temp\ipykernel_23048\1738136020.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query2, conexion)


,ID_C,Name_C,Segment,Country,City,State,Region
0,AA-10315,Alex Avila,Consumer,United States,Round Rock,Texas,Central
1,AA-10375,Allen Armold,Consumer,United States,Providence,Rhode Island,East
2,AA-10480,Andrew Allen,Consumer,United States,Detroit,Michigan,Central
3,AA-10645,Anna Andreadi,Consumer,United States,San Francisco,California,West
4,AB-10015,Aaron Bergman,Consumer,United States,Oklahoma City,Oklahoma,Central


In [6]:
query3= "SELECT * FROM Products"
df3=pd.read_sql(query3, conexion)
df3.head()

C:\Users\mateo\AppData\Local\Temp\ipykernel_23048\2905905273.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df3=pd.read_sql(query3, conexion)


,ID_P,Name_P,Category,Sub_cateogry
0,FUR-BO-10000112,"Bush Birmingham Collection Bookcase, Dark Cherry",Furniture,Bookcases
1,FUR-BO-10000330,"Sauder Camden County Barrister Bookcase, Plank...",Furniture,Bookcases
2,FUR-BO-10000362,Sauder Inglewood Library Bookcases,Furniture,Bookcases
3,FUR-BO-10000468,O'Sullivan 2-Shelf Heavy-Duty Bookcases,Furniture,Bookcases
4,FUR-BO-10000780,O'Sullivan Plantations 2-Door Library in Landv...,Furniture,Bookcases


In [7]:
## Rentabilidad por categoria
query_categoria=" SELECT p.Category as Categorias, SUM(r.Profit) as Total_Ganancias FROM Products p INNER JOIN RegistroVentas r ON p.ID_P=r.Product_ID GROUP BY p.Category ORDER BY Total_Ganancias DESC"
df_categoria=pd.read_sql_query(query_categoria, conexion)
df_categoria

C:\Users\mateo\AppData\Local\Temp\ipykernel_23048\1248087065.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_categoria=pd.read_sql_query(query_categoria, conexion)


,Categorias,Total_Ganancias
0,Technology,6186623.23
1,Office Supplies,4780415.81
2,Furniture,372597.83


In [8]:
## productos con mas perdidas
query='SELECT p.name_p, sum(r.Profit) as Ganancia, sum(r.Quantity) as Cantidad, count(r.Order_ID) as Pedidos FROM RegistroVentas r INNER JOIN Products p on r.Product_ID=p.ID_P WHERE r.Profit<0 GROUP BY P.Name_P ORDER BY Ganancia asc'
df_productos=pd.read_sql_query(query, conexion)
df_productos

C:\Users\mateo\AppData\Local\Temp\ipykernel_23048\102970509.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_productos=pd.read_sql_query(query, conexion)


,name_p,Ganancia,Cantidad,Pedidos
0,Cubify CubeX 3D Printer Triple Head Print,-383999.04,4,1
1,GBC DocuBind P400 Electric Binding System,-315749.68,8,2
2,Ibico EPK-21 Electric Binding System,-292948.45,5,1
3,Bush Advantage Collection Racetrack Conference...,-201923.96,18,4
4,GBC DocuBind TL300 Electric Binding System,-165046.16,8,2
...,...,...,...,...
580,AT&T 841000 Phone,-2.07,2,1
581,Hoover Replacement Belt for Commercial Guardsm...,-1.11,1,1
582,Lexmark 20R1285 X6650 Wireless All-in-One Printer,-0.72,2,1
583,"Eldon 400 Class Desk Accessories, Black Carbon",-0.63,4,1


In [9]:
## Desempeño por region
query="""
    SELECT c.Region, SUM(r.Sales) as Ventas , SUM(r.Profit) as Ganancias, ROUND((SUM(r.Profit)/SUM(r.Sales)),2) as Margen
    FROM RegistroVentas r INNER JOIN Customers c ON r.Customer_ID=c.ID_C
    GROUP BY c.Region
    ORDER BY Ganancias desc
"""
df_region=pd.read_sql(query, conexion)
df_region

C:\Users\mateo\AppData\Local\Temp\ipykernel_23048\2094173388.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_region=pd.read_sql(query, conexion)


,Region,Ventas,Ganancias,Margen
0,West,270425858.0,4104841.20,0.02
1,East,198875241.0,4097389.59,0.02
2,Central,182322448.0,1681359.45,0.01
3,South,125860815.0,1456046.63,0.01


In [16]:
## Productos mas vendidos que generan perdidas
query="""
    SELECT p.Name_P as Producto, SUM(r.Sales) as Ventas, SUM(r.Profit) as Ganancias, COUNT(r.Order_ID) as Ordenes
    FROM RegistroVentas r INNER JOIN Products p ON r.Product_ID=p.ID_P
    GROUP BY p.Name_P
    HAVING SUM(r.Profit) < 0
    ORDER BY Ganancias ASC
"""
df=pd.read_sql_query(query, conexion)
df.head()

C:\Users\mateo\AppData\Local\Temp\ipykernel_23048\112517826.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query(query, conexion)


,Producto,Ventas,Ganancias,Ordenes
0,Cubify CubeX 3D Printer Triple Head Print,300998.0,-383999.04,1
1,Bush Advantage Collection Racetrack Conference...,3188733.0,-201923.96,4
2,Martin Yale Chadless Opener Electric Letter Op...,6662199.0,-130751.17,4
3,Ibico EPK-21 Electric Binding System,4726175.0,-128519.32,2
4,Epson TM-T88V Direct Thermal Printer - Monochr...,692995.0,-93595.95,1


In [ ]:
##conexion.close()